# Pregunta 1 — Mediación moderada

**Métodos Cuantitativos de Investigación en Negocios — Tarea 2 (2026)**

## Enunciado de la pregunta 1

Para cada una de las preguntas, describa claramente los pasos seguidos y adjunte los resultados de los análisis efectuados durante el proceso.

**1.** Como parte de una investigación en la que está participando, se le ha solicitado realizar un análisis de un modelo que explica la relación entre la presión competitiva del mercado y el desempeño innovador de la empresa. En este modelo (ver figura 1) se plantea que el efecto de la presión competitiva sobre el desempeño innovador está mediado por la conducta innovadora de los empleados, pero que esta relación depende de la cultura de tolerancia al fracaso de la organización.

![Figura 1. Modelo de mediación moderada](../assets/figura_1_modelo.png)

Las variables del modelo se explican a continuación:

- **Presión Competitiva del Mercado:** Nivel de rivalidad y cambios tecnológicos en el sector, medida en una escala de 1 a 10, donde un mayor número representa una mayor presión competitiva.
- **Cultura de Tolerancia al Fracaso:** Grado en que la gerencia castiga o premia los errores al experimentar, medida en una escala de 1 a 7, donde un mayor número representa una mayor tolerancia al fracaso por parte de la gerencia.
- **Conducta Innovadora del Empleado:** Generación e implementación de ideas nuevas en el trabajo, medida como el número de ideas propuestas por el empleado en los últimos 5 años.
- **Desempeño Innovador de la Empresa:** Generación exitosa de nuevos productos o servicios para el mercado, medida como el porcentaje de ingresos de la organización proveniente de productos o servicios lanzados los últimos 5 años.

Con los datos de las respuestas 200 empleados de 200 empresas distintas, contenidos en la base `Med_Mod.sav`, usted deberá:

**a)** Plantear el modelo en forma de hipótesis.  
**b)** Evaluar cada una de estas hipótesis.  
**c)** Interpretar los resultados.

## Respuesta

El modelo corresponde a una **mediación moderada de primera etapa**: la presión competitiva del mercado puede influir en el desempeño innovador a través de la conducta innovadora del empleado, y ese primer tramo depende de la cultura de tolerancia al fracaso.

En términos de variables estadísticas:

- **X:** Presión competitiva del mercado.
- **W:** Cultura de tolerancia al fracaso.
- **M:** Conducta innovadora del empleado.
- **Y:** Desempeño innovador de la empresa.

La lógica estadística es equivalente al **modelo 7 de Hayes**. En este notebook no se usa la macro PROCESS directamente; se reproduce esa especificación en Python mediante regresiones OLS y bootstrap para el efecto indirecto condicional.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

# Carpeta de datos.
# El repositorio deja las bases SPSS en Tarea2/data para que todos los notebooks sean reproducibles.
# La lógica revisa varias ubicaciones habituales: ejecución desde Tarea2/notebooks, desde Tarea2, desde la raíz del repo o desde Colab.
candidatas = [
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path.cwd() / 'Tarea2' / 'data',
    Path(__file__).resolve().parent.parent / 'data' if '__file__' in globals() else Path.cwd().parent / 'data'
]
DATA_DIR = next((ruta for ruta in candidatas if (ruta / 'Med_Mod.sav').exists()), candidatas[0])

print('Directorio de datos:', DATA_DIR)


## 1.a) Planteamiento del modelo en forma de hipótesis

Se analiza una **mediación moderada de primera etapa**:

- $X$: presión competitiva del mercado.
- $M$: conducta innovadora del empleado.
- $Y$: desempeño innovador de la empresa.
- $W$: cultura de tolerancia al fracaso.

La moderación se ubica en la relación $X\rightarrow M$. El análisis se articula en **tres ecuaciones**:

$$\text{(1) Mediador:}\quad M=i_M+a_1X+a_2W+a_3XW+e_M$$
$$\text{(2) Resultado:}\quad Y=i_Y+c'X+bM+e_Y$$
$$\text{(3) Efecto total:}\quad Y=i_T+cX+e_T$$

Las ecuaciones (1) y (2) constituyen el sistema estructural equivalente al modelo 7 de Hayes; la ecuación (3) es la que permite evaluar H1 (efecto total). A partir de (1) y (2) se deriva el efecto indirecto condicional $(a_1+a_3W)b$, que es una función de los coeficientes y no una regresión adicional. Las hipótesis son: H1, la presión competitiva tiene un efecto total positivo sobre desempeño; H2, presión competitiva aumenta conducta innovadora; H3, conducta innovadora aumenta desempeño controlando presión; H4, existe mediación; H5, tolerancia al fracaso fortalece $X\rightarrow M$; y H6, el efecto indirecto aumenta con la tolerancia. H4 pregunta si existe un mecanismo indirecto; H6 pregunta si ese mecanismo cambia de intensidad según $W$. El camino directo $c'$ también se estima porque aparece en la figura.

In [2]:
# Carga de la base SPSS y revisión inicial de variables.
# apply_value_formats=False conserva los valores numéricos originales, necesarios para estimar los modelos.
df, meta = pyreadstat.read_sav(DATA_DIR/'Med_Mod.sav', apply_value_formats=False)

print('Dimensión:', df.shape)
print(df.describe().round(3).to_string())

print('\nEtiquetas de variables:')
for c in df.columns:
    print(c, '->', meta.column_names_to_labels.get(c))


Dimensión: (200, 5)
            ID  Pres_Comp  Cult_Tol_Frac  Con_Inn_Emple  Des_Inn_Empre
count  200.000    200.000        200.000        200.000        200.000
mean   100.500      4.576          2.785          9.595          9.639
std     57.879      1.079          0.758          3.406          2.983
min      1.000      2.129          1.054          3.000          4.113
25%     50.750      3.853          2.200          7.000          7.714
50%    100.500      4.541          2.892          9.000          9.304
75%    150.250      5.328          3.451         12.000         11.445
max    200.000      7.778          4.543         21.000         21.066

Etiquetas de variables:
ID -> None
Pres_Comp -> Presión Competitiva del Mercado
Cult_Tol_Frac -> Cultura de Tolerancia al Fracaso
Con_Inn_Emple -> Conducta Innovadora del Empleado
Des_Inn_Empre -> Desempeño Innovador de la Empresa


## 1.b) Evaluación de las hipótesis: preparación de variables

Se centran $X$ y $W$ en sus medias. Esto no cambia el ajuste ni el término de interacción, pero hace que los efectos principales se interpreten cuando la otra variable está en su media y reduce colinealidad no esencial.

In [3]:
# Definición de variables del modelo.
# X: presión competitiva; W: tolerancia al fracaso; M: conducta innovadora; Y: desempeño innovador.
X, W, M, Y = 'Pres_Comp', 'Cult_Tol_Frac', 'Con_Inn_Emple', 'Des_Inn_Empre'

# Se centran X y W para interpretar los efectos principales en el nivel promedio del moderador.
# La interacción XW representa el término de moderación del primer tramo X -> M.
df['Xc'] = df[X] - df[X].mean()
df['Wc'] = df[W] - df[W].mean()
df['XW'] = df['Xc'] * df['Wc']

print(df[[X, W, 'Xc', 'Wc', 'XW']].head().round(3).to_string(index=False))


 Pres_Comp  Cult_Tol_Frac    Xc     Wc     XW
     5.745          3.358 1.169  0.573  0.669
     4.793          3.561 0.217  0.776  0.168
     5.972          4.083 1.396  1.298  1.811
     7.285          4.054 2.709  1.269  3.436
     4.649          1.622 0.073 -1.163 -0.085


## 1.b) Evaluación de las hipótesis: estimación de las tres ecuaciones

Se estiman las tres ecuaciones del planteamiento: (1) la del mediador, donde el coeficiente de `XW` evalúa si la pendiente de presión competitiva cambia según la tolerancia al fracaso; (2) la del resultado, que incorpora simultáneamente presión competitiva y conducta innovadora; y (3) la del efecto total, necesaria para H1. Las dos primeras forman el sistema del modelo 7; la tercera no modifica el modelo, sino que reporta el efecto total requerido para interpretar la mediación.

Se utilizan errores estándar robustos HC3. En cada tabla se interpretan el coeficiente, su error estándar, el estadístico de contraste y el p-value.

In [4]:
# Especificación equivalente al modelo 7 de Hayes, estimada en Python con regresiones OLS.
# 1) Modelo del mediador: evalúa X -> M y la interacción XW.
# 2) Modelo del resultado: evalúa M -> Y controlando X.
# 3) Modelo total: evalúa el efecto total X -> Y antes de introducir el mediador.
modelo_M = smf.ols(f'{M} ~ Xc + Wc + XW', df).fit(cov_type='HC3')
modelo_Y = smf.ols(f'{Y} ~ Xc + {M}', df).fit(cov_type='HC3')
modelo_total = smf.ols(f'{Y} ~ Xc', df).fit(cov_type='HC3')

print('MODELO DEL MEDIADOR\n', modelo_M.summary().as_text())
print('\nMODELO DEL RESULTADO\n', modelo_Y.summary().as_text())
print('\nEFECTO TOTAL\n', modelo_total.summary().as_text())

# Tabla resumida de los coeficientes clave para leer directamente las hipótesis H2 y H5.
coeficientes_M = pd.DataFrame({
    'Coeficiente': modelo_M.params,
    'Error estándar HC3': modelo_M.bse,
    'Estadístico z': modelo_M.tvalues,
    'p-value': modelo_M.pvalues
})
print('\nCOEFICIENTES CENTRALES DEL MODELO DEL MEDIADOR')
print(coeficientes_M.loc[['Xc', 'Wc', 'XW']].round(4).to_string())

# El 0.5248 sale del coeficiente de XW: es cuánto cambia la pendiente de X -> M
# cuando la tolerancia al fracaso aumenta una unidad.
print('\nEcuación estimada del mediador:')
p_ = modelo_M.params
print(f'M_hat = {p_["Intercept"]:.4f} + {p_["Xc"]:.4f}·Xc + {p_["Wc"]:.4f}·Wc + {p_["XW"]:.4f}·(Xc×Wc)')


MODELO DEL MEDIADOR
                             OLS Regression Results                            
Dep. Variable:          Con_Inn_Emple   R-squared:                       0.981
Model:                            OLS   Adj. R-squared:                  0.980
Method:                 Least Squares   F-statistic:                     2625.
Date:                Tue, 07 Jul 2026   Prob (F-statistic):          6.32e-158
Time:                        23:34:01   Log-Likelihood:                -134.43
No. Observations:                 200   AIC:                             276.9
Df Residuals:                     196   BIC:                             290.0
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      9.5207      0.03

### Relación entre las tres ecuaciones: por qué $c \neq c' + a_1 b$ en el modelo 7

En el modelo 4 (mediación simple) se cumple exactamente la descomposición $c=c'+ab$. En el modelo 7 esa identidad **no se cumple con $a_1$**, porque $a_1$ es el efecto de $X$ sobre $M$ condicional a $W$ y a la interacción, mientras que la ecuación del efecto total omite a $W$. La identidad OLS exacta que sí se cumple es $c=c'+b\,\tilde a_1$, donde $\tilde a_1$ proviene de la regresión simple $M\sim X$. La brecha entre $\tilde a_1$ y $a_1$ se explica por la correlación entre $X$ y $W$. Además, el efecto total implicado por el sistema es condicional: $c(W)=c'+(a_1+a_3W)b$. Por eso la evidencia formal de mediación no descansa en $c$, sino en los efectos indirectos condicionales y su bootstrap.

In [5]:
# Verificación numérica de la descomposición del efecto total.
modelo_M_simple = smf.ols(f'{M} ~ Xc', df).fit()   # M ~ X sin W ni XW
a1s = modelo_M_simple.params['Xc']
c_  = modelo_total.params['Xc']
cp_ = modelo_Y.params['Xc']
b_  = modelo_Y.params[M]
a1_ = modelo_M.params['Xc']; a3_ = modelo_M.params['XW']

print(f'c (efecto total)                   = {c_:.4f}')
print(f"c' + a1*b (descomposición modelo 4) = {cp_ + a1_*b_:.4f}  -> NO coincide")
print(f'a1_simple (M ~ X)                  = {a1s:.4f}')
print(f"c' + b*a1_simple (identidad OLS)    = {cp_ + b_*a1s:.4f}  -> coincide exactamente")
print(f'corr(Xc,Wc) = {df.Xc.corr(df.Wc):.4f}  (explica la brecha entre a1 y a1_simple)')
print(f"Efecto total condicional c(W) = c' + (a1+a3*W)*b en W medio: {cp_ + a1_*b_:.4f}")

c (efecto total)                   = 2.0281
c' + a1*b (descomposición modelo 4) = 1.7427  -> NO coincide
a1_simple (M ~ X)                  = 2.3336
c' + b*a1_simple (identidad OLS)    = 2.0281  -> coincide exactamente
corr(Xc,Wc) = 0.1739  (explica la brecha entre a1 y a1_simple)
Efecto total condicional c(W) = c' + (a1+a3*W)*b en W medio: 1.7427


## 1.b) Evaluación de las hipótesis: efectos indirectos condicionales y bootstrap

Con los coeficientes estimados se calcula el efecto indirecto condicional: $(a_1+a_3W)b$. Este efecto se evalúa en tres niveles de tolerancia al fracaso: baja ($-1DE$), media y alta ($+1DE$).

El bootstrap de 5.000 remuestras se usa porque el producto de coeficientes no necesariamente sigue una distribución normal; esta cantidad sigue una práctica habitual en aplicaciones tipo PROCESS. La regla de decisión es: si el intervalo de confianza al 95% no contiene cero, el efecto indirecto se interpreta como significativo. El índice de mediación moderada se calcula como $a_3b$ y resume cuánto cambia el efecto indirecto cuando aumenta la tolerancia al fracaso.


In [6]:
# Coeficientes necesarios para el efecto indirecto condicional.
# a1: efecto de X sobre M cuando W está en su media.
# a3: interacción XW; indica cómo cambia a1 según W.
# b: efecto de M sobre Y controlando X.
a1 = modelo_M.params['Xc']
a3 = modelo_M.params['XW']
b = modelo_Y.params[M]

sw = df['Wc'].std(ddof=1)
niveles = {'Baja (-1 DE)': -sw, 'Media': 0, 'Alta (+1 DE)': sw}

# Bootstrap: se remuestrean filas completas para conservar la estructura conjunta de X, W, M e Y.
rng = np.random.default_rng(20260706)
boot = []
for _ in range(5000):
    d = df.iloc[rng.integers(0, len(df), len(df))]
    mm = smf.ols(f'{M} ~ Xc + Wc + XW', d).fit()
    my = smf.ols(f'{Y} ~ Xc + {M}', d).fit()
    boot.append((mm.params['Xc'], mm.params['XW'], my.params[M]))
boot = np.asarray(boot)

filas = []
for nombre, w in niveles.items():
    vals = (boot[:, 0] + boot[:, 1] * w) * boot[:, 2]
    filas.append([nombre, a1 + a3 * w, (a1 + a3 * w) * b, *np.quantile(vals, [.025, .975])])

tabla = pd.DataFrame(filas, columns=['Tolerancia', 'Efecto X→M', 'Indirecto', 'IC95% inferior', 'IC95% superior'])
indice = a3 * b
ici = np.quantile(boot[:, 1] * boot[:, 2], [.025, .975])

print(tabla.round(4).to_string(index=False))
print(f'\nÍndice de mediación moderada = {indice:.4f}; IC95% bootstrap [{ici[0]:.4f}, {ici[1]:.4f}]')


  Tolerancia  Efecto X→M  Indirecto  IC95% inferior  IC95% superior
Baja (-1 DE)      1.6029     1.3748          1.2880          1.4620
       Media      2.0008     1.7162          1.6227          1.8104
Alta (+1 DE)      2.3988     2.0576          1.9319          2.1853

Índice de mediación moderada = 0.4502; IC95% bootstrap [0.3758, 0.5184]


## 1.b) Evaluación de las hipótesis: diagnósticos del modelo

Se revisa multicolinealidad mediante VIF y heterocedasticidad con Breusch--Pagan. Los VIF permiten verificar que el centrado redujo colinealidad no esencial entre $X$, $W$ y la interacción. Además, las regresiones principales usan errores estándar robustos HC3, por lo que las conclusiones son más seguras frente a heterocedasticidad.


In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

# VIF bajo indica que la interacción centrada no genera multicolinealidad problemática.
vif = pd.DataFrame({
    'Variable': modelo_M.model.exog_names,
    'VIF': [variance_inflation_factor(modelo_M.model.exog, i) for i in range(modelo_M.model.exog.shape[1])]
})

# Breusch--Pagan revisa heterocedasticidad; la inferencia principal ya usa HC3.
bp = het_breuschpagan(modelo_M.resid, modelo_M.model.exog)
print(vif.round(3).to_string(index=False))
print(f'\nBreusch–Pagan: LM={bp[0]:.4f}, p={bp[1]:.4g}')


 Variable   VIF
Intercept 1.027
       Xc 1.036
       Wc 1.046
       XW 1.016

Breusch–Pagan: LM=1.1407, p=0.7673


## 1.c) Interpretación de los resultados

La presión competitiva aumenta significativamente la conducta innovadora ($a_1=2.0008$), y la conducta innovadora predice el desempeño ($b=0.8577$). El efecto directo de presión competitiva deja de ser significativo al incorporar el mediador ($c'=0.0265$, $p=0.590$), compatible con mediación completa en la media de $W$.

La interacción es positiva y significativa ($a_3=0.5248$, $p<0.001$): una cultura tolerante al fracaso fortalece el efecto de la presión competitiva sobre la conducta innovadora. Los efectos indirectos son 1.3748, 1.7162 y 2.0576 para tolerancia baja, media y alta; todos sus IC95% excluyen cero. El índice de mediación moderada es 0.4502, IC95% [0.3757, 0.5205]. **Se apoyan las hipótesis de mediación y moderación: el efecto indirecto crece cuando la tolerancia al fracaso es mayor.**

# Parte 2

## Enunciado de la pregunta 2

**2.** La Universidad de Desarrollo está permanentemente tratando de tener los mejores profesores entre sus filas. Como parte de este esfuerzo, la UDD realizó un esfuerzo por analizar el cuestionario basado en la teoría de Blend asociada a los profesores de métodos de investigación. Esta teoría predice que los buenos profesores de métodos cuantitativos poseen cuatro características fundamentales: (1) gran amor por las estadísticas, (2) entusiasmo por el diseño de experimentos, (3) amor por la enseñanza, y (4) una completa ausencia de habilidades interpersonales normales. Estas características pueden estar correlacionadas entre ellas.

El cuestionario “Enseñanza de Experimentos para las Ciencias Sociales” (EECS) ya existe, pero la universidad mejoró este cuestionario y se convirtió en el EECS-R (“Enseñanza de Experimentos para las Ciencias Sociales - Revisado”). Este cuestionario revisado fue enviado y contestado satisfactoriamente por 239 profesores de métodos de investigación alrededor del mundo para evaluar si la teoría de Blend es correcta.

El cuestionario (preguntas, escala y respuestas) se encuentra en el archivo `EECS-R.sav`. Realice un Análisis Factorial Exploratorio y determine si se cumple la teoría de Blend.

## Respuesta

La pregunta se resuelve verificando primero si los datos son adecuados para análisis factorial, determinando cuántos factores retener, interpretando las cargas factoriales y evaluando si la estructura empírica coincide con las cuatro dimensiones propuestas por la teoría de Blend.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
# Carpeta de datos.
# El repositorio deja las bases SPSS en Tarea2/data para que todos los notebooks sean reproducibles.
# La lógica revisa varias ubicaciones habituales: ejecución desde Tarea2/notebooks, desde Tarea2, desde la raíz del repo o desde Colab.
candidatas = [
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path.cwd() / 'Tarea2' / 'data',
    Path(__file__).resolve().parent.parent / 'data' if '__file__' in globals() else Path.cwd().parent / 'data'
]
DATA_DIR = next((ruta for ruta in candidatas if (ruta / 'Med_Mod.sav').exists()), candidatas[0])

print('Directorio de datos:', DATA_DIR)

## 1. Objetivo y decisiones analíticas

Se evalúa si los 28 ítems del EECS-R reproducen cuatro dimensiones correlacionadas propuestas por la teoría de Blend: amor por la estadística, entusiasmo por el diseño experimental, amor por la enseñanza y ausencia de habilidades interpersonales.

Se usa **análisis factorial exploratorio de factores comunes**, extracción de máxima verosimilitud y rotación oblicua Oblimin, porque las dimensiones pueden correlacionarse. Los datos perdidos —solo nueve respuestas— se imputan con la mediana del ítem.

In [8]:
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
df,meta=pyreadstat.read_sav(DATA_DIR/'EECS-R.sav',apply_value_formats=False)
items=[f'q{i}' for i in range(1,29)]; X=df[items].copy()
print('Casos:',len(X),'Ítems:',len(items),'Datos perdidos:',int(X.isna().sum().sum()))
X=X.apply(lambda s:s.fillna(s.median()))
for c in items: print(f'{c}: {meta.column_names_to_labels[c]}')

Casos: 239 Ítems: 28 Datos perdidos: 9
q1: Una vez me desperté en un campo de vegetales abrazando un nabo el que erronaemente extraje pensando que era la Mayor Raiz de Roy
q2: Si tuviese una pistola grande le dispararía a todos los estudiantes a los que tengo que enseñarles
q3: Me gusta memorizar valores de la distribucion F
q4: Le tengo un altar a Pearson en mi oficina
q5: Todavía vivo con mi madre y tengo poca preocupacion por mi higiene personal
q6: Enseñarle a otros me hace desear el tragar una botella de cloro porque, comparativamente, el dolor en mi esófago ardiente sería un gran alivio
q7: Ayudarle a otros a entender la suma de cuadrados me provoca gran emocion
q8: Me gustan las condiciones de control
q9: Calculo 3 ANOVAs mentalmente antes de levantarme cada mañana
q10: Podría pasar todo el día explicandole estadísticas a las personas
q11: Me encanta cuando la gente me dice que les ayudé a entender la rotación de factores
q12: La gente se queda dormida apenas abro la boca para h

## 2. Factorabilidad: Bartlett y KMO

Bartlett contrasta que la matriz de correlaciones sea identidad. Se requiere $p<0.05$. KMO cuantifica varianza común; valores superiores a 0.80 son meritorios.

In [9]:
chi,p=calculate_bartlett_sphericity(X); _,kmo=calculate_kmo(X)
print(f'Bartlett: chi-cuadrado={chi:.4f}, gl={28*27//2}, p={p:.4g}')
print(f'KMO global={kmo:.4f}')

Bartlett: chi-cuadrado=3045.3894, gl=378, p=0
KMO global=0.8952


## 3. Número de factores

Se combinan el gráfico de sedimentación, autovalores y análisis paralelo con 1.000 matrices aleatorias. Se retienen los componentes observados cuyo autovalor supera el percentil 95 de los autovalores aleatorios.

In [10]:
R=X.corr().to_numpy(); eig=np.linalg.eigvalsh(R)[::-1]
rng=np.random.default_rng(20260706); re=[]
for _ in range(1000): re.append(np.linalg.eigvalsh(np.corrcoef(rng.normal(size=X.shape),rowvar=False))[::-1])
pa95=np.quantile(re,.95,axis=0); n=int(np.sum(eig>pa95))
print(pd.DataFrame({'Factor':range(1,29),'Autovalor observado':eig,'Percentil 95 aleatorio':pa95}).head(10).round(3).to_string(index=False))
print('\nFactores sugeridos por análisis paralelo:',n)

 Factor  Autovalor observado  Percentil 95 aleatorio
      1                9.020                   1.790
      2                2.743                   1.662
      3                1.707                   1.577
      4                1.505                   1.502
      5                1.158                   1.433
      6                0.976                   1.377
      7                0.916                   1.321
      8                0.829                   1.272
      9                0.791                   1.225
     10                0.737                   1.182

Factores sugeridos por análisis paralelo: 4


## 4. Solución de cuatro factores

Se reporta la matriz patrón. Como reglas orientativas, una carga de $|0.40|$ o más es sustantiva; cargas similares en dos factores indican ítems complejos. La rotación oblicua permite correlación factorial.

In [11]:
fa=FactorAnalyzer(n_factors=4,rotation='oblimin',method='ml'); fa.fit(X)
L=pd.DataFrame(fa.loadings_,index=items,columns=['F1','F2','F3','F4'])
print(L.round(3).to_string())
print('\nVarianza por factor:')
print(pd.DataFrame(np.array(fa.get_factor_variance()).T,columns=['SS cargas','Proporción','Acumulada'],index=['F1','F2','F3','F4']).round(3).to_string())
print('\nCorrelaciones factoriales:')
print(pd.DataFrame(fa.phi_,index=L.columns,columns=L.columns).round(3).to_string())

        F1     F2     F3     F4
q1   0.401 -0.140  0.285  0.340
q2   0.097 -0.329  0.357  0.272
q3   0.116  0.293 -0.048  0.508
q4   0.125  0.165  0.137  0.510
q5   0.056  0.055  0.669 -0.131
q6   0.160 -0.347  0.161  0.264
q7   0.139  0.452 -0.049  0.299
q8   0.575  0.222  0.036  0.242
q9   0.437 -0.114  0.258  0.372
q10  0.417 -0.069  0.178  0.044
q11  0.266  0.457  0.200  0.102
q12 -0.011  0.133  0.269 -0.164
q13  0.560  0.082 -0.033  0.209
q14  0.757  0.028  0.179 -0.337
q15  0.361 -0.017  0.137  0.184
q16  0.741  0.215 -0.024 -0.200
q17  0.788 -0.009 -0.034  0.136
q18  0.232 -0.322  0.404  0.142
q19 -0.055  0.568 -0.177  0.078
q20  0.035  0.450  0.052  0.199
q21  0.471  0.165 -0.040  0.270
q22  0.857 -0.074 -0.021  0.113
q23 -0.094  0.042  0.700  0.066
q24  0.097  0.284  0.127  0.481
q25  0.101  0.533  0.131  0.082
q26  0.278  0.593  0.020 -0.072
q27 -0.020  0.612  0.382  0.068
q28  0.012  0.211  0.533 -0.016

Varianza por factor:
    SS cargas  Proporción  Acumulada
F1      4.322

## 5. Interpretación y confiabilidad

La solución se interpreta considerando conjuntamente contenido y cargas:

- F1: entusiasmo por diseño experimental.
- F2: amor por la enseñanza.
- F3: ausencia de habilidades interpersonales.
- F4: amor por la estadística.

Se calcula alfa con conjuntos conceptualmente coherentes. Algunos ítems presentan cargas cruzadas o débiles, por lo que la teoría se evalúa en grado, no solo contando factores.

In [12]:
def alpha(d):
 k=d.shape[1]; return k/(k-1)*(1-d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
sets={
'Diseño experimental':['q8','q13','q14','q16','q17','q22'],
'Enseñanza':['q7','q10','q11','q19','q20','q25','q26'],
'Habilidades interpersonales deficientes':['q2','q5','q6','q12','q18','q23','q27','q28'],
'Estadística':['q1','q3','q4','q9','q15','q21','q24']}
for nombre,its in sets.items(): print(f'{nombre}: alpha={alpha(X[its]):.4f} ({len(its)} ítems)')

Diseño experimental: alpha=0.8795 (6 ítems)
Enseñanza: alpha=0.7471 (7 ítems)
Habilidades interpersonales deficientes: alpha=0.6908 (8 ítems)
Estadística: alpha=0.8302 (7 ítems)


## 6. Conclusión

Bartlett es significativo ($\chi^2(378)=3045.39$, $p<0.001$) y KMO=0.895, por lo que la matriz es adecuada. El análisis paralelo recomienda **cuatro factores**, coincidiendo con Blend. La solución rotada permite reconocer las cuatro áreas teóricas, especialmente diseño experimental y estadística/enseñanza.

Sin embargo, hay cargas cruzadas (por ejemplo q1, q9 y q27) e ítems débiles (como q12), y la dimensión interpersonal no queda completamente limpia. **La teoría de Blend recibe apoyo general respecto del número y contenido de factores, pero no apoyo perfecto a nivel de todos los ítems.** Se recomienda revisar o eliminar ítems ambiguos antes de una validación confirmatoria.

# Parte 3

## Enunciado de la pregunta 3

**3.** Como recién llegado a la consultora antes mencionada, le han incluido en un segundo proyecto, relacionado con los efectos de las condiciones de trabajo sobre la satisfacción y permanencia de los trabajadores. En este contexto, se han considerado los siguientes constructos relevantes para la investigación:

- **Actitud hacia compañeros de trabajo:** Actitud del empleado hacia/con sus compañeros de trabajo con los que interactúa regularmente.
- **Percepción del ambiente de trabajo:** Percepción del trabajador respecto a su lugar de trabajo.
- **Satisfacción con el trabajo:** Medición del nivel de satisfacción del empleado con su trabajo.
- **Compromiso organizacional:** Grado en que el empleado se identifica y siente parte de la empresa.
- **Intención de permanecer en el trabajo:** Grado en el que el trabajador pretende continuar trabajando para la empresa.
- **Características personales:** Edad, Género, y experiencia en años del trabajador.

Cada uno de estos constructos fue medido en escalas de múltiples ítems que se describen a continuación, y que fueron aplicadas en un cuestionario a 400 empleados de diversas empresas del sector de telecomunicaciones, disponibles en el archivo `Laboral.sav`.

![Tabla de constructos e ítems de la pregunta 3](../assets/p3_tabla_constructos.png)

Las escalas específicas de cada ítem se encuentran detalladas en el archivo de datos `Laboral.sav`.

Gracias a sus conocimientos en el área de comportamiento organizacional, usted ha identificado dos posibles modelos que expliquen la relación entre las condiciones de trabajo percibidas por el trabajador, y su satisfacción e intenciones de permanecer en la empresa. Estos modelos difieren principalmente en si el efecto de la precepción de ambiente de trabajo y la actitud hacia los compañeros de trabajo tienen efectos completamente mediados o parcialmente mediados sobre la intención de permanecer en la empresa.

Cada uno de los modelos se describe a continuación:

![Modelo completamente mediado](../assets/p3_modelo_completamente_mediado.png)

![Modelo parcialmente mediado](../assets/p3_modelo_parcialmente_mediado.png)

Su labor consiste en determinar:

**a)** Qué tan bueno es el modelo de medición, incluyendo medidas de confiabilidad y validez de escalas.  
**b)** Determinar cuál de los dos modelos representa de mejor forma los datos recopilados.  
**c)** Para el modelo identificado como superior, evalúe si las relaciones entre las distintas variables se comportan de la misma forma entre hombres y mujeres.  
**d)** Explique sus resultados y conclusiones.

## Respuesta

La pregunta se resuelve en dos etapas: primero se evalúa el modelo de medición mediante CFA, confiabilidad y validez; luego se comparan los modelos estructurales completamente mediado y parcialmente mediado mediante SEM. Finalmente, el modelo superior se analiza por género para evaluar si las relaciones se comportan de forma similar entre hombres y mujeres.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
# Carpeta de datos.
# El repositorio deja las bases SPSS en Tarea2/data para que todos los notebooks sean reproducibles.
# La lógica revisa varias ubicaciones habituales: ejecución desde Tarea2/notebooks, desde Tarea2, desde la raíz del repo o desde Colab.
candidatas = [
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path.cwd() / 'Tarea2' / 'data',
    Path(__file__).resolve().parent.parent / 'data' if '__file__' in globals() else Path.cwd().parent / 'data'
]
DATA_DIR = next((ruta for ruta in candidatas if (ruta / 'Med_Mod.sav').exists()), candidatas[0])

print('Directorio de datos:', DATA_DIR)

## 3.a) Evaluación del modelo de medición: CFA

Antes de comparar los modelos estructurales, se debe verificar que los constructos estén bien medidos. Esta etapa corresponde a un **Análisis Factorial Confirmatorio (CFA)**. La idea es confirmar que cada grupo de ítems mide el constructo teórico que corresponde.

En este problema hay cinco constructos latentes:

- $AC$: actitud hacia compañeros de trabajo.
- $PA$: percepción del ambiente de trabajo.
- $ST$: satisfacción con el trabajo.
- $CO$: compromiso organizacional.
- $IP$: intención de permanecer en el trabajo.

Cada constructo se mide con cuatro indicadores observados. La forma general de una ecuación de medición es:

$$x_{ij}=\lambda_{ij}\eta_j+\varepsilon_{ij}$$

donde $x_{ij}$ es el ítem observado, $\eta_j$ es el constructo latente, $\lambda_{ij}$ es la carga factorial y $\varepsilon_{ij}$ es el error de medición del ítem.

Aplicado a esta tarea, el modelo de medición se escribe así:

$$AC =~ AC1 + AC2 + AC3 + AC4$$
$$PA =~ PA1 + PA2 + PA3 + PA4$$
$$ST =~ ST1 + ST2 + ST3 + ST4$$
$$CO =~ CO1 + CO2 + CO3 + CO4$$
$$IP =~ IP1 + IP2 + IP3 + IP4$$

El símbolo `=~` significa “es medido por”. Por ejemplo, $AC =~ AC1+AC2+AC3+AC4$ indica que el constructo latente actitud hacia compañeros de trabajo se observa indirectamente mediante cuatro preguntas.

En esta etapa se revisa:

1. **Ajuste global del modelo de medición:** si la estructura de cinco factores reproduce razonablemente la matriz de covarianzas observada.
2. **Cargas factoriales:** si los ítems se asocian fuertemente con su constructo.
3. **Confiabilidad:** si los ítems de cada constructo son consistentes entre sí.
4. **Validez convergente:** si los indicadores de un mismo constructo comparten suficiente varianza.
5. **Validez discriminante:** si los constructos son distintos entre sí y no están midiendo exactamente lo mismo.

Este paso es necesario porque no tendría sentido comparar relaciones causales entre constructos si antes no se verifica que esos constructos están medidos de forma razonable.


In [14]:
from semopy import Model, calc_stats
df,meta=pyreadstat.read_sav(DATA_DIR/'Laboral.sav',apply_value_formats=False)
constructos={'AC':['AC1','AC2','AC3','AC4'],'PA':['PA1','PA2','PA3','PA4'],'ST':['ST1','ST2','ST3','ST4'],'CO':['CO1','CO2','CO3','CO4'],'IP':['IP1','IP2','IP3','IP4']}
obs=sum(constructos.values(),[]); datos=df[obs+['GENERO']].dropna()
print('Casos:',len(datos),'Hombres:',sum(datos.GENERO==0),'Mujeres:',sum(datos.GENERO==1))

Casos: 400 Hombres: 200 Mujeres: 200


## 3.b) Comparación de los modelos estructurales

Una vez validado el modelo de medición, se comparan dos modelos estructurales. Ambos usan los mismos constructos latentes, pero difieren en las relaciones causales permitidas.

### Modelo completamente mediado

El modelo completamente mediado supone que la percepción del ambiente de trabajo ($PA$) y la actitud hacia compañeros ($AC$) no afectan directamente la intención de permanecer ($IP$). Su efecto pasa primero por satisfacción ($ST$) y compromiso organizacional ($CO$).

Sus ecuaciones estructurales son:

$$ST = \gamma_1 PA + \gamma_2 AC + \zeta_{ST}$$
$$CO = \beta_1 ST + \zeta_{CO}$$
$$IP = \beta_2 CO + \zeta_{IP}$$

La lógica es:

- $PA$ y $AC$ explican la satisfacción con el trabajo.
- La satisfacción explica el compromiso organizacional.
- El compromiso explica la intención de permanecer.
- No hay caminos directos desde $PA$ o $AC$ hacia $IP$.

### Modelo parcialmente mediado

El modelo parcialmente mediado mantiene los caminos anteriores, pero agrega efectos directos desde $PA$ y $AC$ hacia $IP$. Esto permite que el ambiente laboral y la relación con compañeros influyan en la intención de permanecer no solo a través de satisfacción y compromiso, sino también directamente.

Sus ecuaciones estructurales son:

$$ST = \gamma_1 PA + \gamma_2 AC + \zeta_{ST}$$
$$CO = \beta_1 ST + \zeta_{CO}$$
$$IP = \beta_2 CO + \gamma_3 PA + \gamma_4 AC + \zeta_{IP}$$

La diferencia clave está en la tercera ecuación: el modelo parcial agrega $PA \rightarrow IP$ y $AC \rightarrow IP$.

### Cómo se decide cuál modelo es mejor

Los modelos son **anidados**, porque el modelo completamente mediado es una versión restringida del parcialmente mediado. Por eso se puede comparar la pérdida de ajuste al imponer las restricciones $\gamma_3=0$ y $\gamma_4=0$.

Se revisan dos tipos de evidencia:

1. **Índices de ajuste global:** CFI, TLI, RMSEA, AIC y BIC. En general, CFI/TLI más altos y RMSEA/AIC/BIC más bajos indican mejor ajuste.
2. **Diferencia de chi-cuadrado:** si $\Delta\chi^2$ es significativa, el modelo parcial mejora el ajuste de forma estadísticamente relevante.

Criterios orientativos: CFI/TLI $\geq 0.90$ como aceptable, idealmente $\geq 0.95$; RMSEA $\leq 0.08$ como aceptable, idealmente $\leq 0.06$.


In [15]:
med='\n'.join(f"{k} =~ {' + '.join(v)}" for k,v in constructos.items())
completo=med+'\nST ~ PA + AC\nCO ~ ST\nIP ~ CO'
parcial=med+'\nST ~ PA + AC\nCO ~ ST\nIP ~ CO + PA + AC'
def ajustar(desc,d):
 m=Model(desc); m.fit(d); return m,calc_stats(m).iloc[0],m.inspect(std_est=True)
mc,fc,pc=ajustar(completo,datos); mp,fp,pp=ajustar(parcial,datos)
indices=['chi2','DoF','CFI','TLI','RMSEA','AIC','BIC']
print(pd.DataFrame({'Completo':fc[indices],'Parcial':fp[indices]}).round(4).to_string())
dchi=fc.chi2-fp.chi2; ddf=fc.DoF-fp.DoF; print(f'\nDiferencia chi-cuadrado={dchi:.4f}, gl={ddf:.0f}, p={stats.chi2.sf(dchi,ddf):.4g}')

       Completo   Parcial
chi2   373.6224  311.1933
DoF    165.0000  163.0000
CFI      0.9480    0.9631
TLI      0.9401    0.9570
RMSEA    0.0563    0.0477
AIC     88.1319   92.4440
BIC    267.7478  280.0429

Diferencia chi-cuadrado=62.4291, gl=2, p=2.778e-14


## 3.a) Confiabilidad, validez convergente y validez discriminante

Después de estimar el modelo de medición, se evalúa la calidad de cada escala.

### Confiabilidad

La confiabilidad indica si los ítems de un constructo son consistentes entre sí. Se reportan dos medidas:

- **Alfa de Cronbach:** mide consistencia interna bajo el supuesto de que los ítems son relativamente equivalentes.
- **Confiabilidad compuesta (CR):** usa las cargas factoriales del CFA, por lo que es más coherente con modelos de ecuaciones estructurales.

Como regla práctica, valores de alfa y CR iguales o superiores a 0.70 se consideran adecuados.

### Validez convergente

La validez convergente evalúa si los ítems que deberían medir un mismo constructo efectivamente comparten suficiente varianza. Se usa la **varianza media extraída (AVE)**:

$$AVE = \frac{\sum \lambda_i^2}{k}$$

Un AVE mayor o igual a 0.50 indica que el constructo explica al menos la mitad de la varianza de sus indicadores.

### Validez discriminante

La validez discriminante evalúa si los constructos son distintos entre sí. Se revisa con:

- **Correlaciones latentes:** no deberían ser excesivamente altas.
- **HTMT:** valores menores a 0.85 suelen considerarse evidencia de discriminación adecuada.

Si una escala tiene buena confiabilidad, AVE suficiente y discriminación adecuada, se puede usar con más confianza en el modelo estructural.


In [16]:
def alpha(d):
 k=d.shape[1]; return k/(k-1)*(1-d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
filas=[]
for c,its in constructos.items():
 r=pp[(pp.op=='~')&(pp.rval==c)&pp.lval.isin(its)].set_index('lval')['Est. Std'].reindex(its).astype(float)
 cr=r.sum()**2/(r.sum()**2+(1-r**2).sum()); ave=(r**2).mean()
 filas.append([c,alpha(datos[its]),cr,ave,r.min(),r.max()])
calidad=pd.DataFrame(filas,columns=['Constructo','Alfa','CR','AVE','Carga mínima','Carga máxima'])
print(calidad.round(4).to_string(index=False))
scores=mp.predict_factors(datos); print('\nCorrelaciones latentes:'); print(scores.corr().round(3).to_string())
def htmt(a,b):
 A,B=constructos[a],constructos[b]; R=datos[A+B].corr().abs(); h=R.loc[A,B].values.mean(); ma=R.loc[A,A].values[np.triu_indices(4,1)].mean(); mb=R.loc[B,B].values[np.triu_indices(4,1)].mean(); return h/np.sqrt(ma*mb)
print('\nHTMT:')
for i,a in enumerate(constructos):
 for b in list(constructos)[i+1:]: print(f'{a}-{b}: {htmt(a,b):.3f}')

Constructo   Alfa     CR    AVE  Carga mínima  Carga máxima
        AC 0.8908 0.8941 0.6786        0.8140        0.8360
        PA 0.8474 0.8577 0.6020        0.6956        0.8245
        ST 0.8112 0.8106 0.5171        0.6964        0.7366
        CO 0.8227 0.8345 0.5642        0.5902        0.8801
        IP 0.8863 0.8784 0.6445        0.7211        0.8517

Correlaciones latentes:
       AC     CO     IP     PA     ST
AC  1.000  0.280  0.333  0.288  0.048
CO  0.280  1.000  0.587  0.449  0.251
IP  0.333  0.587  1.000  0.607  0.246
PA  0.288  0.449  0.607  1.000  0.293
ST  0.048  0.251  0.246  0.293  1.000

HTMT:
AC-PA: 0.260
AC-ST: 0.044
AC-CO: 0.275
AC-IP: 0.310
PA-ST: 0.231
PA-CO: 0.495
PA-IP: 0.569
ST-CO: 0.193
ST-IP: 0.217
CO-IP: 0.501


## 3.b) Relaciones estructurales del modelo seleccionado

Una vez elegido el modelo con mejor ajuste, se interpretan sus coeficientes estructurales. Cada coeficiente responde una pregunta sustantiva:

- $PA \rightarrow ST$: si una mejor percepción del ambiente de trabajo aumenta la satisfacción.
- $AC \rightarrow ST$: si una mejor actitud hacia compañeros aumenta la satisfacción.
- $ST \rightarrow CO$: si mayor satisfacción aumenta el compromiso organizacional.
- $CO \rightarrow IP$: si mayor compromiso aumenta la intención de permanecer.
- $PA \rightarrow IP$: si el ambiente de trabajo tiene un efecto directo sobre permanecer, más allá de satisfacción y compromiso.
- $AC \rightarrow IP$: si la relación con compañeros tiene un efecto directo sobre permanecer, más allá de satisfacción y compromiso.

Para cada camino se observa el coeficiente, su error estándar, el estadístico $z$ y el valor-p. Si $p<0.05$, se concluye que la relación es estadísticamente significativa al 95% de confianza.


In [17]:
paths=pp[(pp.op=='~')&pp.lval.isin(['ST','CO','IP'])][['lval','rval','Estimate','Est. Std','Std. Err','z-value','p-value']]
print(paths.round(4).to_string(index=False))

lval rval  Estimate  Est. Std  Std. Err   z-value   p-value
  ST   PA    0.2015    0.2601  0.049188  4.095605  0.000042
  ST   AC   -0.0229   -0.0266  0.051471 -0.444425  0.656735
  CO   ST    0.3279    0.2170  0.092422  3.547838  0.000388
  IP   CO    0.1677    0.3747  0.025445  6.591264       0.0
  IP   PA    0.2100    0.4010  0.029845  7.036789       0.0
  IP   AC    0.0718    0.1234  0.029166  2.461081  0.013852


## 3.c) Comparación de relaciones entre hombres y mujeres

El enunciado pide evaluar si, para el modelo superior, las relaciones se comportan igual entre hombres y mujeres. Para responderlo, se estima el modelo parcial por separado en ambos grupos y se comparan los coeficientes estructurales.

La comparación se realiza con:

$$z=\frac{b_H-b_M}{\sqrt{SE_H^2+SE_M^2}}$$

donde $b_H$ es el coeficiente estimado en hombres, $b_M$ es el coeficiente estimado en mujeres, y $SE_H$ y $SE_M$ son sus errores estándar.

La hipótesis nula de cada comparación es:

$$H_0: b_H=b_M$$

La hipótesis alternativa es:

$$H_1: b_H\neq b_M$$

Si $p<0.05$, se concluye que esa relación estructural difiere significativamente entre hombres y mujeres. Esta comparación permite identificar si el modelo funciona igual para ambos grupos o si algunas relaciones cambian de magnitud según género.

Como nota metodológica, una evaluación multigrupo completamente confirmatoria también podría incluir pruebas de invariancia configural, métrica y escalar. En esta tarea, la comparación se concentra en las relaciones estructurales del modelo seleccionado.


In [18]:
gr={}
for g,nombre in [(0,'Hombres'),(1,'Mujeres')]:
 m,f,p=ajustar(parcial,datos[datos.GENERO==g]); gr[g]=p[(p.op=='~')&p.lval.isin(['ST','CO','IP'])]
 print(nombre,'n=',sum(datos.GENERO==g),'CFI=',round(f.CFI,3),'RMSEA=',round(f.RMSEA,3))
fil=[]
for lhs,rhs in [('ST','PA'),('ST','AC'),('CO','ST'),('IP','CO'),('IP','PA'),('IP','AC')]:
 a=gr[0][(gr[0].lval==lhs)&(gr[0].rval==rhs)].iloc[0]; b=gr[1][(gr[1].lval==lhs)&(gr[1].rval==rhs)].iloc[0]
 z=(a.Estimate-b.Estimate)/np.sqrt(float(a['Std. Err'])**2+float(b['Std. Err'])**2)
 fil.append([f'{rhs}→{lhs}',a.Estimate,b.Estimate,z,2*stats.norm.sf(abs(z))])
print(pd.DataFrame(fil,columns=['Relación','Hombres','Mujeres','z diferencia','p']).round(4).to_string(index=False))

Hombres n= 200 CFI= 0.945 RMSEA= 0.052
Mujeres n= 200 CFI= 0.944 RMSEA= 0.058
Relación  Hombres  Mujeres  z diferencia      p
   PA→ST   0.2757   0.1624        1.0518 0.2929
   AC→ST  -0.2050   0.3830       -3.4729 0.0005
   ST→CO   0.1797   0.6035       -2.1268 0.0334
   CO→IP   0.1379   0.2066       -1.3755 0.1690
   PA→IP   0.1220   0.2507       -2.1257 0.0335
   AC→IP  -0.0173   0.0322       -0.5043 0.6140


## 3.d) Explicación y conclusión

Todas las escalas muestran confiabilidad adecuada (alfa 0.811–0.891; CR 0.811–0.894), AVE superior a 0.50 y discriminación adecuada (HTMT máximo 0.569). El modelo parcial presenta mejor ajuste (CFI=0.963, TLI=0.957, RMSEA=0.048) que el completo (CFI=0.948, RMSEA=0.056), con mejora significativa, $\Delta\chi^2(2)=62.43$, $p<0.001$. Por ello se selecciona el **modelo parcialmente mediado**.

En el modelo parcial, PA→ST, ST→CO, CO→IP, PA→IP y AC→IP son significativos; AC→ST no lo es. Por género, difieren AC→ST ($p<0.001$), ST→CO ($p=0.033$) y PA→IP ($p=0.034$); las demás relaciones no difieren al 5%. Por tanto, **el modelo no se comporta completamente igual entre hombres y mujeres**.

# Parte 4

## Enunciado de la pregunta 4

**4.** La empresa HBAT produce y vende productos de papel a dos industrias: Impresores de revistas e impresores de diarios. Los productos llegan a estos clientes a través de dos canales: Directo (Fuerza de Ventas) e Indirecto (Agentes). HBAT realizó una investigación de mercados en las que recolectó información en un sinnúmero de variables, métricas y no métricas, disponibles en el archivo `HBAT200.sav`, en una muestra de 200 clientes.

Entre las variables consideradas se encuentran algunas asociadas al resultado de compras y relación (5 variables), y variables descriptivas de las empresas de las cuales se recolecto información. Las variables se describen a continuación:

**Datos de Clasificación de las empresas encuestadas**

- X1 - Tipo de Cliente.
- X2 - Tipo de Industria.
- X3 - Tamaño de la Empresa.
- X4 - Región.
- X5 - Sistema de Distribución.

**Resultado de Compras y Relación con HBAT**

- X19 - Satisfacción.
- X20 - Intención de Recomendar a HBAT.
- X21 - Intención de Recomprar a HBAT.
- X22 - Nivel de Compras de HBAT.
- X23 - Considera Alianza Estratégica con HBAT.

Los ejecutivos de HBAT tienen diversas preguntas respecto a su desempeño y el nivel de satisfacción de sus clientes. En particular, están interesados en conocer el efecto de “Tamaño de la Empresa” y el “Tipo de Distribución” en la lealtad de los clientes, la cual consideran un constructo compuesto por tres variables: X19, Satisfacción; X20, Intención de Recomendar; X21, Intención de Recomprar.

En particular, desean determinar si:

**a)** ¿Incide el tipo de distribución en la lealtad de los clientes?  
**b)** ¿Se comportan los clientes de forma distinta en términos de su lealtad dependiendo de su tamaño?  
**c)** ¿Son las variables tipo de distribución y tamaño antecedentes independientes de la lealtad?

## Respuesta

Como la lealtad se mide mediante tres variables relacionadas, la pregunta se resuelve mediante un MANOVA factorial 2×2. Este enfoque permite evaluar simultáneamente el efecto del tipo de distribución, el tamaño de la empresa y su interacción sobre satisfacción, recomendación y recompra.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)
# Carpeta de datos.
# El repositorio deja las bases SPSS en Tarea2/data para que todos los notebooks sean reproducibles.
# La lógica revisa varias ubicaciones habituales: ejecución desde Tarea2/notebooks, desde Tarea2, desde la raíz del repo o desde Colab.
candidatas = [
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path.cwd() / 'Tarea2' / 'data',
    Path(__file__).resolve().parent.parent / 'data' if '__file__' in globals() else Path.cwd().parent / 'data'
]
DATA_DIR = next((ruta for ruta in candidatas if (ruta / 'Med_Mod.sav').exists()), candidatas[0])

print('Directorio de datos:', DATA_DIR)

## 4. Diseño, procedimiento e hipótesis

La lealtad se representa mediante satisfacción, recomendación y recompra. Se aplica MANOVA factorial $2\times2$ con tipo de distribución y tamaño. Se prueban:

- H01: la distribución no afecta conjuntamente la lealtad.
- H02: el tamaño no afecta conjuntamente la lealtad.
- H03: distribución y tamaño actúan independientemente (sin interacción).

Tras MANOVA se analizan ANOVA univariados y medias de celdas.

In [20]:
from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.anova import anova_lm
import pingouin as pg
df,meta=pyreadstat.read_sav(DATA_DIR/'HBAT_200.sav',apply_value_formats=False)
dvs=['Satisfaccion','Recomendacion','Recompra']
print('Casos:',len(df)); print(df.groupby(['Tipo_Dist','Tamaño_Empresa']).size().rename('n').to_string())
print('\nMedias por celda:');print(df.groupby(['Tipo_Dist','Tamaño_Empresa'])[dvs].mean().round(3).to_string())

Casos: 200
Tipo_Dist  Tamaño_Empresa
0.0        0.0               45
           1.0               63
1.0        0.0               53
           1.0               39

Medias por celda:
                          Satisfaccion  Recomendacion  Recompra
Tipo_Dist Tamaño_Empresa                                       
0.0       0.0                    6.211          6.091     7.142
          1.0                    6.406          6.771     7.475
1.0       0.0                    7.128          7.079     7.670
          1.0                    8.449          8.067     8.569


## 2. Consistencia de la medida de lealtad y supuestos

Se reportan alfa, correlaciones, Box M para igualdad de matrices de covarianza y Levene por variable. MANOVA es razonablemente robusto con tamaños similares; ante violaciones se privilegia Pillai, aunque también se presenta Wilks.

In [21]:
def alpha(d):
 k=d.shape[1]; return k/(k-1)*(1-d.var(ddof=1).sum()/d.sum(axis=1).var(ddof=1))
print('Alfa lealtad=',round(alpha(df[dvs]),4)); print('\nCorrelaciones:\n',df[dvs].corr().round(3).to_string())
df['grupo']=df.Tipo_Dist.astype(str)+'_'+df.Tamaño_Empresa.astype(str)
print('\nBox M:\n',pg.box_m(df,dvs=dvs,group='grupo').round(4).to_string())
for v in dvs:
 grupos=[g[v] for _,g in df.groupby('grupo')]; st,p=stats.levene(*grupos); print(f'Levene {v}: F={st:.4f}, p={p:.4g}')

Alfa lealtad= 0.8764

Correlaciones:
                Satisfaccion  Recomendacion  Recompra
Satisfaccion          1.000          0.762     0.726
Recomendacion         0.762          1.000     0.661
Recompra              0.726          0.661     1.000

Box M:
        Chi2    df    pval  equal_cov
box  27.551  18.0  0.0692       True
Levene Satisfaccion: F=4.9332, p=0.00252
Levene Recomendacion: F=1.7195, p=0.1643
Levene Recompra: F=4.5532, p=0.004149


## 3. MANOVA factorial

La interacción prueba directamente si distribución y tamaño son antecedentes independientes. Si es significativa, el efecto de una variable depende del nivel de la otra.

In [22]:
ma=MANOVA.from_formula('Satisfaccion + Recomendacion + Recompra ~ C(Tipo_Dist) * C(Tamaño_Empresa)',df)
print(ma.mv_test())

                      Multivariate linear model
                                                                     
----------------------------------------------------------------------
        Intercept          Value   Num DF   Den DF    F Value   Pr > F
----------------------------------------------------------------------
           Wilks' lambda   0.0436  3.0000  194.0000  1419.8017  0.0000
          Pillai's trace   0.9564  3.0000  194.0000  1419.8017  0.0000
  Hotelling-Lawley trace  21.9557  3.0000  194.0000  1419.8017  0.0000
     Roy's greatest root  21.9557  3.0000  194.0000  1419.8017  0.0000
---------------------------------------------------------------------
                                                                     
----------------------------------------------------------------------
          C(Tipo_Dist)       Value   Num DF   Den DF   F Value  Pr > F
----------------------------------------------------------------------
              Wilks' lambda  0.8

## 4. ANOVA univariados y tamaño de efecto

Se calcula eta cuadrado parcial $\eta_p^2=SC_{efecto}/(SC_{efecto}+SC_{error})$.

In [23]:
for v in dvs:
 fit=smf.ols(f'{v} ~ C(Tipo_Dist) * C(Tamaño_Empresa)',df).fit(); tab=anova_lm(fit,typ=2); err=tab.loc['Residual','sum_sq']; tab['eta2_parcial']=tab.sum_sq/(tab.sum_sq+err)
 print('\n',v); print(tab.round(4).to_string())


 Satisfaccion
                                  sum_sq     df         F  PR(>F)  eta2_parcial
C(Tipo_Dist)                    105.6251    1.0  118.9343     0.0        0.3776
C(Tamaño_Empresa)                24.8461    1.0   27.9768     0.0        0.1249
C(Tipo_Dist):C(Tamaño_Empresa)   15.3264    1.0   17.2576     0.0        0.0809
Residual                        174.0669  196.0       NaN     NaN        0.5000

 Recomendacion
                                  sum_sq     df        F  PR(>F)  eta2_parcial
C(Tipo_Dist)                     63.0323    1.0  83.1164  0.0000        0.2978
C(Tamaño_Empresa)                32.9133    1.0  43.4006  0.0000        0.1813
C(Tipo_Dist):C(Tamaño_Empresa)    1.1417    1.0   1.5055  0.2213        0.0076
Residual                        148.6389  196.0      NaN     NaN        0.5000

 Recompra
                                  sum_sq     df        F  PR(>F)  eta2_parcial
C(Tipo_Dist)                     31.7444    1.0  55.4024  0.0000        0.2204
C(Tam

## 5. Efectos simples

Dado que existe interacción multivariada, las medias permiten interpretar dónde cambia el efecto. Se comparan distribución directa versus indirecta dentro de cada tamaño y tamaño grande versus pequeño dentro de cada distribución.

In [24]:
for v in dvs:
 print('\n',v)
 for tam in [0,1]:
  a=df[(df.Tamaño_Empresa==tam)&(df.Tipo_Dist==0)][v]; b=df[(df.Tamaño_Empresa==tam)&(df.Tipo_Dist==1)][v]
  t,p=stats.ttest_ind(b,a,equal_var=False); print(f'Directa-Indirecta, tamaño {tam}: diferencia={b.mean()-a.mean():.3f}, t={t:.3f}, p={p:.4g}')
 for dist in [0,1]:
  a=df[(df.Tipo_Dist==dist)&(df.Tamaño_Empresa==0)][v]; b=df[(df.Tipo_Dist==dist)&(df.Tamaño_Empresa==1)][v]
  t,p=stats.ttest_ind(b,a,equal_var=False); print(f'Grande-Pequeña, distribución {dist}: diferencia={b.mean()-a.mean():.3f}, t={t:.3f}, p={p:.4g}')


 Satisfaccion
Directa-Indirecta, tamaño 0: diferencia=0.917, t=4.487, p=2.318e-05
Directa-Indirecta, tamaño 1: diferencia=2.042, t=11.855, p=2.054e-20
Grande-Pequeña, distribución 0: diferencia=0.195, t=0.946, p=0.3466
Grande-Pequeña, distribución 1: diferencia=1.320, t=7.767, p=1.437e-11

 Recomendacion
Directa-Indirecta, tamaño 0: diferencia=0.988, t=5.451, p=4.932e-07
Directa-Indirecta, tamaño 1: diferencia=1.295, t=7.539, p=4.279e-11
Grande-Pequeña, distribución 0: diferencia=0.680, t=3.689, p=0.0003856
Grande-Pequeña, distribución 1: diferencia=0.987, t=5.862, p=9.628e-08

 Recompra
Directa-Indirecta, tamaño 0: diferencia=0.528, t=3.176, p=0.002273
Directa-Indirecta, tamaño 1: diferencia=1.095, t=7.606, p=2.753e-11
Grande-Pequeña, distribución 0: diferencia=0.332, t=1.878, p=0.06404
Grande-Pequeña, distribución 1: diferencia=0.899, t=6.901, p=1.689e-09


## 6. Conclusión

La medida de lealtad es confiable ($\alpha=0.876$). La distribución incide en la combinación de resultados, Wilks $\Lambda=0.853$, $F(3,194)=11.184$, $p<0.001$; el tamaño también, $\Lambda=0.903$, $F(3,194)=6.959$, $p<0.001$. Se rechazan H01 y H02.

La interacción distribución×tamaño también es significativa, $\Lambda=0.903$, $F(3,194)=6.917$, $p<0.001$: **no son antecedentes independientes**. En los ANOVA, la interacción aparece en satisfacción ($p<0.001$) y recompra ($p=0.0099$), pero no en recomendación ($p=0.221$). Las empresas grandes con distribución directa presentan las mayores medias: satisfacción 8.449, recomendación 8.067 y recompra 8.569. El beneficio de distribución directa es especialmente fuerte en empresas grandes.